# 1 - Model Contracts

This notebook develops the model-contract layer for KalmanWorks: `ProcessModel`, `MeasurementModel`, optional control coupling, and the assembled `StateSpaceModel`. Once this section is stable and reviewed, it can be promoted into the Python package under `implementations/python/src/kalmanworks/`.


In [ ]:
from __future__ import annotations

from abc import ABC, abstractmethod
from typing import TypeAlias, Any

import numpy as np
from numpy.typing import NDArray, ArrayLike

from dataclasses import dataclass

import inspect

import matplotlib.pyplot as plt

In [ ]:
# ======================================================================
# Type aliases
# ======================================================================
StateVector       : TypeAlias = NDArray[np.float64]
MeasurementVector : TypeAlias = NDArray[np.float64]
ControlVector     : TypeAlias = NDArray[np.float64]
Matrix            : TypeAlias = NDArray[np.float64]
SampleMatrix      : TypeAlias = NDArray[np.float64]
SigmaPointMatrix  : TypeAlias = NDArray[np.float64]
WeightVector      : TypeAlias = NDArray[np.float64]


## 1.1 - `ProcessModel`

In [ ]:
class ProcessModel(ABC):
    """
    Contract for discrete-time state dynamics and process noise.

    State propagation
    -----------------
    This model describes how the state evolves from time step k to the
    next time step k+1.

    Linear form
    -----------
    {x_k+1} = [F_k] {x_k} + [G_k] {u_k} + {w_k}

    Nonlinear form
    --------------
    {x_k+1} = f({x_k}, {u_k}, dt) + {w_k}

    Process noise
    -------------
    The process noise covariance is

    [Q_k] = E[{w_k} {w_k^T}]

    where:
    - {x_k} is the state at time step k
    - {x_k+1} is the propagated state at the next time step
    - {u_k} is the control input at time step k
    - {w_k} is the process noise at time step k
    - [F_k] is the linear state-transition matrix
    - [G_k] is the linear control-input matrix
    - [Q_k] is the discrete-time process noise covariance

    Usage
    -----
    Subclass `ProcessModel` to define how the system state propagates from
    one discrete time step to the next.

    Typical pattern:
    - define `state_labels` for the ordered components of the state vector
    - define `control_labels` if the process model uses named control inputs
    - implement `Q(...)`
    - implement `F(...)` for linear filters, or `f(...)` / `F_jacobian(...)`
    for nonlinear filters as needed
    - implement `G(...)` when the process model includes a linear
    control-input mapping
    - override `apply_state_delta(...)` and `state_residual(...)` when the
    state does not live in ordinary Euclidean space, such as quaternion
    or angle-valued states

    A `ProcessModel` instance is typically registered inside a
    `StateSpaceModel` and used by the filter predict step.

    Required
    --------
    - state_labels
    - Q(...)

    Optional / estimator-dependent
    ------------------------------
    - control_labels
    - f(...)
    - F(...)
    - F_jacobian(...)
    - G(...)

    Notes
    -----
    Geometry hooks are provided for states that are not represented in
    ordinary Euclidean space.
    """
    # =========================================================================
    # Constructor
    # =========================================================================
    def __init__(
        self,
        state_labels: tuple[str, ...],
        control_labels: tuple[str, ...] | None = None,
    ) -> None:
        """
        Store the ordered state labels and optional control labels.
        """
        self._state_labels = tuple(state_labels)
        self._control_labels = None if control_labels is None else tuple(control_labels)

        self._validate_labels(self._state_labels, name="state_labels")

        if self._control_labels is not None:
            self._validate_labels(self._control_labels, name="control_labels")

    # =========================================================================
    # Public properties
    # =========================================================================
    @property
    def state_labels(self) -> tuple[str, ...]:
        """
        Return the ordered state labels that define the state vector layout.
        """
        return self._state_labels

    @property
    def control_labels(self) -> tuple[str, ...] | None:
        """
        Return the ordered control labels, or None if no named control input exists.
        """
        return self._control_labels

    @property
    def state_dim(self) -> int:
        """
        Return the state dimension derived from state_labels.
        """
        return len(self._state_labels)

    @property
    def control_dim(self) -> int | None:
        """
        Return the control dimension derived from control_labels, if available.
        """
        return None if self._control_labels is None else len(self._control_labels)

    # =========================================================================
    # Required stochastic interface
    # =========================================================================
    @abstractmethod
    def Q(
        self,
        dt: float,
        x: StateVector | None = None,
        u: ControlVector | None = None,
    ) -> Matrix:
        """
        Return the discrete-time process noise covariance.
        """
        raise NotImplementedError

    # =========================================================================
    # Estimator-dependent capabilities
    # =========================================================================
    def f(
        self,
        x: StateVector,
        u: ControlVector | None = None,
        dt: float | None = None,
    ) -> StateVector:
        """
        Propagate the state with the nonlinear process model.
        """
        raise NotImplementedError(f"{type(self).__name__} must implement f().")

    def F(
        self,
        dt: float,
        x: StateVector | None = None,
        u: ControlVector | None = None,
    ) -> Matrix:
        """
        Return the linear state-transition matrix.
        """
        raise NotImplementedError(f"{type(self).__name__} must implement F().")

    def F_jacobian(
        self,
        x: StateVector,
        u: ControlVector | None = None,
        dt: float | None = None,
    ) -> Matrix:
        """
        Return the Jacobian of f with respect to state.
        """
        raise NotImplementedError(f"{type(self).__name__} must implement F_jacobian().")

    def G(
        self,
        dt: float,
        x: StateVector | None = None,
        u: ControlVector | None = None,
    ) -> Matrix:
        """
        Return the linear control-input matrix when the model uses one.
        """
        raise NotImplementedError(f"{type(self).__name__} must implement G().")

    # =========================================================================
    # Geometry hooks
    # =========================================================================
    def state_residual(
        self,
        a: StateVector,
        b: StateVector,
    ) -> StateVector:
        """
        Return the residual a - b in state space.

        Default behavior assumes Euclidean state arithmetic. Override this
        for non-Euclidean states such as quaternion or angle-valued states.
        """
        state_a = np.asarray(a, dtype=np.float64)
        state_b = np.asarray(b, dtype=np.float64)
        return state_a - state_b

    def state_mean(
        self,
        samples: SampleMatrix,
        weights: WeightVector,
    ) -> StateVector:
        """
        Return the weighted mean in state space.

        Default behavior assumes Euclidean state arithmetic. Override this
        for non-Euclidean states when sample-based averaging is required.
        """
        state_samples = np.asarray(samples, dtype=np.float64)
        sample_weights = np.asarray(weights, dtype=np.float64)
        return np.average(state_samples, axis=0, weights=sample_weights)

    def apply_state_delta(
        self,
        x: StateVector,
        dx: StateVector,
    ) -> StateVector:
        """
        Apply a correction in state space.

        Default behavior assumes Euclidean state arithmetic. Override this
        for non-Euclidean states such as quaternion or angle-valued states.
        """
        state = np.asarray(x, dtype=np.float64)
        delta = np.asarray(dx, dtype=np.float64)
        return state + delta

    # =========================================================================
    # Internal helpers
    # =========================================================================
    @staticmethod
    def _validate_labels(labels: tuple[str, ...], name: str) -> None:
        """
        Validate a tuple of ordered labels used to define a vector layout.
        """
        if len(labels) == 0:
            raise ValueError(f"{name} must be non-empty.")

        if any(not isinstance(label, str) for label in labels):
            raise TypeError(f"{name} must contain only strings.")

        if any(label == "" for label in labels):
            raise ValueError(f"{name} must not contain empty labels.")

        if any(label != label.strip() for label in labels):
            raise ValueError(f"{name} must not contain leading or trailing whitespace.")

        if any(not label.isidentifier() for label in labels):
            raise ValueError(
                f"{name} must contain valid Python identifiers so named builders "
                f"can use keyword arguments."
            )

        if len(set(labels)) != len(labels):
            raise ValueError(f"{name} must contain unique labels.")

## 1.2 - `MeasurementModel`

In [ ]:
class MeasurementModel(ABC):
    """
    Contract for mapping state into measurement space, measurement noise,
    and measurement geometry.

    Measurement mapping
    -------------------
    This model maps the current state {x_k} into measurement space {z_k}.

    Linear form
    -----------
    {z_k} = [H_k] {x_k} + {v_k}

    Nonlinear form
    --------------
    {z_k} = h({x_k}, dt) + {v_k}

    Measurement noise
    -----------------
    The measurement noise covariance is

    [R_k] = E[{v_k} {v_k}^T]

    where:
    - {x_k} is the state at time step k
    - {z_k} is the measurement at time step k
    - {v_k} is the measurement noise at time step k
    - [H_k] is the linear measurement matrix
    - [R_k] is the measurement noise covariance

    Usage
    -----
    Subclass `MeasurementModel` to define how one measurement source maps
    the current state into measurement space.

    Typical pattern:
    - define `state_labels` to match the process/state layout
    - define `measurement_labels` for the components of the measurement vector
    - implement `R(...)`
    - implement `H(...)` for linear filters, or `h(...)` / `H_jacobian(...)`
    for nonlinear filters as needed
    - override `innovation(...)` when the measurement does not live in
    ordinary Euclidean space, such as wrapped angles, bearings, or
    attitude-like quantities

    A `MeasurementModel` instance is typically registered inside a
    `StateSpaceModel` and selected by name during the filter update step.

    Required
    --------
    - state_labels
    - measurement_labels
    - R(...)

    Optional / estimator-dependent
    ------------------------------
    - h(...)
    - H(...)
    - H_jacobian(...)

    Notes
    -----
    Geometry hooks are provided for measurements that are not represented
    in ordinary Euclidean space.
    """

    # =========================================================================
    # Constructor
    # =========================================================================
    def __init__(
        self,
        state_labels: tuple[str, ...],
        measurement_labels: tuple[str, ...],
    ) -> None:
        """
        Store the ordered state and measurement labels used by this model.
        """
        self._state_labels = tuple(state_labels)
        self._measurement_labels = tuple(measurement_labels)

        self._validate_labels(self._state_labels, name="state_labels")
        self._validate_labels(self._measurement_labels, name="measurement_labels")

    # =========================================================================
    # Public properties
    # =========================================================================
    @property
    def state_labels(self) -> tuple[str, ...]:
        """
        Return the ordered state labels expected by this model.
        """
        return self._state_labels

    @property
    def measurement_labels(self) -> tuple[str, ...]:
        """
        Return the ordered measurement labels produced by this model.
        """
        return self._measurement_labels

    @property
    def state_dim(self) -> int:
        """
        Return the state dimension derived from state_labels.
        """
        return len(self._state_labels)

    @property
    def measurement_dim(self) -> int:
        """
        Return the measurement dimension derived from measurement_labels.
        """
        return len(self._measurement_labels)

    # =========================================================================
    # Required stochastic interface
    # =========================================================================
    @abstractmethod
    def R(
        self,
        x: StateVector | None = None,
        z: MeasurementVector | None = None,
        dt: float | None = None,
    ) -> Matrix:
        """
        Return the measurement noise covariance.
        """
        raise NotImplementedError

    # =========================================================================
    # Estimator-dependent capabilities
    # =========================================================================
    def h(
        self,
        x: StateVector,
        dt: float | None = None,
    ) -> MeasurementVector:
        """
        Predict the measurement from the current state.
        """
        raise NotImplementedError(f"{type(self).__name__} must implement h().")

    def H(
        self,
        x: StateVector | None = None,
        dt: float | None = None,
    ) -> Matrix:
        """
        Return the linear measurement matrix.
        """
        raise NotImplementedError(f"{type(self).__name__} must implement H().")

    def H_jacobian(
        self,
        x: StateVector,
        dt: float | None = None,
    ) -> Matrix:
        """
        Return the Jacobian of h with respect to state.
        """
        raise NotImplementedError(f"{type(self).__name__} must implement H_jacobian().")

    # =========================================================================
    # Geometry hooks
    # =========================================================================
    def innovation(
        self,
        z: MeasurementVector,
        z_pred: MeasurementVector,
    ) -> MeasurementVector:
        """
        Return the measurement residual between z and z_pred.

        Default behavior assumes Euclidean measurement arithmetic.
        Override this for non-Euclidean measurements such as wrapped
        angles, bearings, azimuth/elevation, or attitude-like quantities.
        """
        measurement = np.asarray(z, dtype=np.float64)
        predicted_measurement = np.asarray(z_pred, dtype=np.float64)
        return measurement - predicted_measurement

    def measurement_mean(
        self,
        samples: SampleMatrix,
        weights: WeightVector,
    ) -> MeasurementVector:
        """
        Return the weighted mean of multiple measurement-space samples.

        Default behavior assumes Euclidean measurement arithmetic.
        Override this when sample-based measurement averaging must respect
        non-Euclidean measurement geometry.
        """
        measurement_samples = np.asarray(samples, dtype=np.float64)
        sample_weights = np.asarray(weights, dtype=np.float64)
        return np.average(measurement_samples, axis=0, weights=sample_weights)

    # =========================================================================
    # Internal helpers
    # =========================================================================
    @staticmethod
    def _validate_labels(labels: tuple[str, ...], name: str) -> None:
        """
        Validate a tuple of ordered labels used to define a vector layout.
        """
        if len(labels) == 0:
            raise ValueError(f"{name} must be non-empty.")

        if any(not isinstance(label, str) for label in labels):
            raise TypeError(f"{name} must contain only strings.")

        if any(label == "" for label in labels):
            raise ValueError(f"{name} must not contain empty labels.")

        if any(label != label.strip() for label in labels):
            raise ValueError(f"{name} must not contain leading or trailing whitespace.")

        if any(not label.isidentifier() for label in labels):
            raise ValueError(
                f"{name} must contain valid Python identifiers so named builders "
                f"can use keyword arguments."
            )

        if len(set(labels)) != len(labels):
            raise ValueError(f"{name} must contain unique labels.")


## 1.3 - `StateSpaceModel`

In [ ]:
class StateSpaceModel:
    """
    Assemble one process model and zero or more measurement models into
    one estimator-facing system contract.

    System composition
    ------------------
    This class combines:
    - one `ProcessModel`
    - zero or more named `MeasurementModel` instances

    so that filters can interact with one unified interface for:
    - state propagation
    - process noise
    - measurement prediction
    - measurement noise
    - geometry-aware state and measurement operations

    Usage
    -----
    Create one `StateSpaceModel` by supplying:
    - one `ProcessModel` instance
    - zero or more named `MeasurementModel` instances

    Typical pattern:
    - define the system dynamics in a `ProcessModel`
    - define one `MeasurementModel` per measurement source or measurement type
    - register measurement models by name
    - pass this assembled object to the filter as the single model interface

    This class is a composition and delegation layer. Concrete physics
    should live in `ProcessModel` and `MeasurementModel` subclasses.

    When exactly one measurement model is registered, measurement-facing
    methods may omit `measurement_name`. When multiple measurement models
    are registered, `measurement_name` must be provided explicitly.

    Notes
    -----
    Geometry-sensitive methods such as `apply_state_delta(...)`,
    `state_residual(...)`, `state_mean(...)`, `innovation(...)`, and
    `measurement_mean(...)` should be authored by the component models.
    Filters call them through this assembled interface so Euclidean
    defaults and non-Euclidean overrides both flow through one
    system-level contract.
    """

    # =========================================================================
    # Constructor
    # =========================================================================
    def __init__(
        self,
        process_model: ProcessModel,
        measurement_models: dict[str, MeasurementModel] | None = None,
    ) -> None:
        """
        Construct an assembled system model.
        """
        self.process_model = process_model
        self.measurement_models: dict[str, MeasurementModel] = {}

        self._state_labels = tuple(self.process_model.state_labels)
        self._state_index_map = self._build_index_map(self._state_labels)

        self._control_labels = self.process_model.control_labels
        self._control_index_map = (
            {}
            if self._control_labels is None
            else self._build_index_map(self._control_labels)
        )

        self._measurement_index_maps: dict[str, dict[str, int]] = {}

        if measurement_models is not None:
            for name, model in measurement_models.items():
                self.add_measurement_model(name, model)

    # =========================================================================
    # Public properties
    # =========================================================================
    @property
    def state_labels(self) -> tuple[str, ...]:
        """
        Return the canonical state labels.
        """
        return self._state_labels

    @property
    def state_dim(self) -> int:
        """
        Return the canonical state dimension.
        """
        return len(self._state_labels)

    @property
    def control_labels(self) -> tuple[str, ...] | None:
        """
        Return the control labels, or None if no named control input exists.
        """
        return self._control_labels

    @property
    def control_dim(self) -> int | None:
        """
        Return the control dimension, or None if no named control input exists.
        """
        return None if self._control_labels is None else len(self._control_labels)

    @property
    def measurement_names(self) -> tuple[str, ...]:
        """
        Return the registered measurement model names.
        """
        return tuple(self.measurement_models.keys())

    @property
    def has_measurement_models(self) -> bool:
        """
        Return True if at least one measurement model is registered.
        """
        return len(self.measurement_models) > 0

    @property
    def has_control_input(self) -> bool:
        """
        Return True if the process model defines named control inputs.
        """
        return self._control_labels is not None

    # =========================================================================
    # Public model registration and metadata
    # =========================================================================
    def add_measurement_model(
        self,
        name: str,
        model: MeasurementModel,
    ) -> None:
        """
        Register a named measurement model.
        """
        if not isinstance(name, str) or not name.strip():
            raise ValueError("Measurement model name must be a non-empty string.")

        if name in self.measurement_models:
            raise ValueError(f"Measurement model '{name}' is already registered.")

        if tuple(model.state_labels) != self._state_labels:
            raise ValueError(
                f"Measurement model '{name}' state_labels must match process model "
                "state_labels exactly."
            )

        self.measurement_models[name] = model
        self._measurement_index_maps[name] = self._build_index_map(
            tuple(model.measurement_labels)
        )

    def has_measurement_model(self, name: str) -> bool:
        """
        Return True if a named measurement model is registered.
        """
        return name in self.measurement_models

    def get_measurement_model(
        self,
        measurement_name: str | None = None,
    ) -> MeasurementModel:
        """
        Return one registered measurement model.
        """
        resolved_name = self._resolve_measurement_name(measurement_name)
        return self.measurement_models[resolved_name]

    def get_measurement_labels(
        self,
        measurement_name: str | None = None,
    ) -> tuple[str, ...]:
        """
        Return the measurement labels for one measurement model.
        """
        return tuple(self.get_measurement_model(measurement_name).measurement_labels)

    def measurement_dim(
        self,
        measurement_name: str | None = None,
    ) -> int:
        """
        Return the dimension of one measurement model.
        """
        return self.get_measurement_model(measurement_name).measurement_dim

    # =========================================================================
    # Public index helpers
    # =========================================================================
    def state_index(self, label: str) -> int:
        """
        Return the index of a named state.
        """
        if label not in self._state_index_map:
            raise KeyError(f"State label '{label}' is not defined.")
        return self._state_index_map[label]

    def state_indices(self, *labels: str) -> tuple[int, ...]:
        """
        Return the indices of one or more named states.
        """
        return tuple(self.state_index(label) for label in labels)

    def control_index(self, label: str) -> int:
        """
        Return the index of a named control input.
        """
        self._require_control_input()

        if label not in self._control_index_map:
            raise KeyError(f"Control label '{label}' is not defined.")
        return self._control_index_map[label]

    def control_indices(self, *labels: str) -> tuple[int, ...]:
        """
        Return the indices of one or more named control inputs.
        """
        return tuple(self.control_index(label) for label in labels)

    def measurement_index(
        self,
        measurement_name: str | None,
        label: str,
    ) -> int:
        """
        Return the index of a named measurement within one measurement model.
        """
        resolved_name = self._resolve_measurement_name(measurement_name)
        index_map = self._measurement_index_maps[resolved_name]

        if label not in index_map:
            raise KeyError(
                f"Measurement label '{label}' is not defined for model '{resolved_name}'."
            )

        return index_map[label]

    def measurement_indices(
        self,
        measurement_name: str | None,
        *labels: str,
    ) -> tuple[int, ...]:
        """
        Return the indices of one or more named measurements within one model.
        """
        return tuple(self.measurement_index(measurement_name, label) for label in labels)

    # =========================================================================
    # Public builders
    # =========================================================================
    def zero_state(self) -> StateVector:
        """
        Return a zero state vector.
        """
        return np.zeros(self.state_dim, dtype=np.float64)

    def state(self, **values: float) -> StateVector:
        """
        Build a full state vector from named values.
        """
        return self._build_named_vector(
            index_map=self._state_index_map,
            values=values,
            vector_name="state",
        )

    def zero_control(self) -> ControlVector:
        """
        Return a zero control vector.
        """
        self._require_control_input()
        assert self.control_dim is not None
        return np.zeros(self.control_dim, dtype=np.float64)

    def control(self, **values: float) -> ControlVector:
        """
        Build a full control vector from named values.
        """
        self._require_control_input()
        return self._build_named_vector(
            index_map=self._control_index_map,
            values=values,
            vector_name="control",
        )

    def zero_measurement(
        self,
        measurement_name: str | None = None,
    ) -> MeasurementVector:
        """
        Return a zero measurement vector for one measurement model.
        """
        measurement_model = self.get_measurement_model(measurement_name)
        return np.zeros(measurement_model.measurement_dim, dtype=np.float64)

    def measurement(
        self,
        measurement_name: str | None = None,
        **values: float,
    ) -> MeasurementVector:
        """
        Build a full measurement vector from named values.
        """
        resolved_name = self._resolve_measurement_name(measurement_name)
        return self._build_named_vector(
            index_map=self._measurement_index_maps[resolved_name],
            values=values,
            vector_name=f"measurement '{resolved_name}'",
        )

    def zero_state_covariance(self) -> Matrix:
        """
        Return a zero state covariance matrix.
        """
        return np.zeros((self.state_dim, self.state_dim), dtype=np.float64)

    def state_covariance(self, **variances: float) -> Matrix:
        """
        Build a diagonal state covariance matrix from named variances.
        """
        diagonal = self._build_named_vector(
            index_map=self._state_index_map,
            values=variances,
            vector_name="state covariance",
        )

        if np.any(diagonal < 0.0):
            raise ValueError("State covariance variances must be non-negative.")

        return np.diag(diagonal).astype(np.float64)

    # =========================================================================
    # Process-model delegation
    # =========================================================================
    def f(
        self,
        x: StateVector,
        u: ControlVector | None = None,
        dt: float | None = None,
    ) -> StateVector:
        """
        Delegate nonlinear state propagation to the process model.
        """
        x_next = self.process_model.f(x=x, u=u, dt=dt)
        return self._coerce_vector_output(
            vector=x_next,
            expected_dim=self.state_dim,
            method_name="f(...)",
            vector_name="state vector",
        )

    def F(
        self,
        dt: float,
        x: StateVector | None = None,
        u: ControlVector | None = None,
    ) -> Matrix:
        """
        Delegate the linear state-transition matrix to the process model.
        """
        return self.process_model.F(dt=dt, x=x, u=u)

    def F_jacobian(
        self,
        x: StateVector,
        u: ControlVector | None = None,
        dt: float | None = None,
    ) -> Matrix:
        """
        Delegate the process Jacobian to the process model.
        """
        return self.process_model.F_jacobian(x=x, u=u, dt=dt)

    def Q(
        self,
        dt: float,
        x: StateVector | None = None,
        u: ControlVector | None = None,
    ) -> Matrix:
        """
        Delegate the process noise covariance to the process model.
        """
        return self.process_model.Q(dt=dt, x=x, u=u)

    def G(
        self,
        dt: float,
        x: StateVector | None = None,
        u: ControlVector | None = None,
    ) -> Matrix:
        """
        Delegate the linear control-input matrix to the process model.
        """
        return self.process_model.G(dt=dt, x=x, u=u)

    def state_residual(
        self,
        a: StateVector,
        b: StateVector,
    ) -> StateVector:
        """
        Return the state-space residual through the process model.
        """
        residual = self.process_model.state_residual(a=a, b=b)
        return self._coerce_vector_output(
            vector=residual,
            expected_dim=self.state_dim,
            method_name="state_residual(...)",
            vector_name="state vector",
        )

    def state_mean(
        self,
        samples: SampleMatrix,
        weights: WeightVector,
    ) -> StateVector:
        """
        Return the weighted state mean through the process model.
        """
        mean_state = self.process_model.state_mean(samples=samples, weights=weights)
        return self._coerce_vector_output(
            vector=mean_state,
            expected_dim=self.state_dim,
            method_name="state_mean(...)",
            vector_name="state vector",
        )

    def apply_state_delta(
        self,
        x: StateVector,
        dx: StateVector,
    ) -> StateVector:
        """
        Return the corrected state through the process model.
        """
        updated_state = self.process_model.apply_state_delta(x=x, dx=dx)
        return self._coerce_vector_output(
            vector=updated_state,
            expected_dim=self.state_dim,
            method_name="apply_state_delta(...)",
            vector_name="state vector",
        )

    # =========================================================================
    # Measurement-model delegation
    # =========================================================================
    def h(
        self,
        x: StateVector,
        measurement_name: str | None = None,
        dt: float | None = None,
    ) -> MeasurementVector:
        """
        Delegate nonlinear measurement prediction to one measurement model.
        """
        measurement_model = self.get_measurement_model(measurement_name)
        z_pred = measurement_model.h(x=x, dt=dt)
        return self._coerce_vector_output(
            vector=z_pred,
            expected_dim=measurement_model.measurement_dim,
            method_name="h(...)",
            vector_name="measurement vector",
            context_name=self._resolve_measurement_name(measurement_name),
        )

    def H(
        self,
        x: StateVector | None = None,
        measurement_name: str | None = None,
        dt: float | None = None,
    ) -> Matrix:
        """
        Delegate the linear measurement matrix to one measurement model.
        """
        measurement_model = self.get_measurement_model(measurement_name)
        return measurement_model.H(x=x, dt=dt)

    def H_jacobian(
        self,
        x: StateVector,
        measurement_name: str | None = None,
        dt: float | None = None,
    ) -> Matrix:
        """
        Delegate the measurement Jacobian to one measurement model.
        """
        measurement_model = self.get_measurement_model(measurement_name)
        return measurement_model.H_jacobian(x=x, dt=dt)

    def R(
        self,
        x: StateVector | None = None,
        z: MeasurementVector | None = None,
        measurement_name: str | None = None,
        dt: float | None = None,
    ) -> Matrix:
        """
        Delegate the measurement noise covariance to one measurement model.
        """
        measurement_model = self.get_measurement_model(measurement_name)
        return measurement_model.R(x=x, z=z, dt=dt)

    def innovation(
        self,
        z: MeasurementVector,
        z_pred: MeasurementVector,
        measurement_name: str | None = None,
    ) -> MeasurementVector:
        """
        Return the measurement residual through one measurement model.
        """
        resolved_name = self._resolve_measurement_name(measurement_name)
        innovation_vector = self.measurement_models[resolved_name].innovation(
            z=z,
            z_pred=z_pred,
        )
        return self._coerce_vector_output(
            vector=innovation_vector,
            expected_dim=self.measurement_models[resolved_name].measurement_dim,
            method_name="innovation(...)",
            vector_name="measurement vector",
            context_name=resolved_name,
        )

    def measurement_mean(
        self,
        samples: SampleMatrix,
        weights: WeightVector,
        measurement_name: str | None = None,
    ) -> MeasurementVector:
        """
        Return the weighted measurement mean through one measurement model.
        """
        resolved_name = self._resolve_measurement_name(measurement_name)
        mean_measurement = self.measurement_models[resolved_name].measurement_mean(
            samples=samples,
            weights=weights,
        )
        return self._coerce_vector_output(
            vector=mean_measurement,
            expected_dim=self.measurement_models[resolved_name].measurement_dim,
            method_name="measurement_mean(...)",
            vector_name="measurement vector",
            context_name=resolved_name,
        )

    # =========================================================================
    # Internal helpers
    # =========================================================================
    @staticmethod
    def _build_index_map(labels: tuple[str, ...]) -> dict[str, int]:
        """
        Build a label-to-index lookup table.
        """
        return {label: index for index, label in enumerate(labels)}

    @staticmethod
    def _build_named_vector(
        index_map: dict[str, int],
        values: dict[str, float],
        vector_name: str,
    ) -> NDArray[np.float64]:
        """
        Build one full named vector from a label-to-index map and value mapping.
        """
        expected_labels = set(index_map.keys())
        provided_labels = set(values.keys())

        unknown_labels = provided_labels - expected_labels
        if unknown_labels:
            unknown_labels_text = ", ".join(sorted(unknown_labels))
            raise KeyError(f"Unknown labels for {vector_name}: {unknown_labels_text}")

        missing_labels = expected_labels - provided_labels
        if missing_labels:
            missing_labels_text = ", ".join(sorted(missing_labels))
            raise KeyError(f"Missing labels for {vector_name}: {missing_labels_text}")

        vector = np.empty(len(index_map), dtype=np.float64)
        for label, raw_value in values.items():
            vector[index_map[label]] = float(raw_value)

        return vector

    def _resolve_measurement_name(
        self,
        measurement_name: str | None,
    ) -> str:
        """
        Resolve one measurement name or raise a clear error.
        """
        if measurement_name is not None:
            if measurement_name not in self.measurement_models:
                raise KeyError(f"Measurement model '{measurement_name}' is not registered.")
            return measurement_name

        if not self.measurement_models:
            raise ValueError("No measurement models are registered in this StateSpaceModel.")

        if len(self.measurement_models) > 1:
            available_models = ", ".join(self.measurement_names)
            raise ValueError(
                "measurement_name must be provided when multiple measurement models "
                f"are registered. Available models: {available_models}."
            )

        return next(iter(self.measurement_models))

    def _require_control_input(self) -> None:
        """
        Raise if the process model defines no named control input.
        """
        if self._control_labels is None:
            raise ValueError("No named control input is defined by this process model.")

    @staticmethod
    def _coerce_vector_output(
        vector,
        expected_dim: int,
        method_name: str,
        vector_name: str,
        context_name: str | None = None,
    ) -> NDArray[np.float64]:
        """
        Coerce and validate one delegated vector output.
        """
        output_vector = np.atleast_1d(np.asarray(vector, dtype=np.float64))

        if output_vector.ndim != 1:
            if context_name is None:
                raise ValueError(f"{method_name} must return a 1D {vector_name}.")
            raise ValueError(
                f"{method_name} for '{context_name}' must return a 1D {vector_name}."
            )

        expected_shape = (expected_dim,)
        if output_vector.shape != expected_shape:
            if context_name is None:
                raise ValueError(
                    f"{method_name} must return shape {expected_shape}, "
                    f"got {output_vector.shape}."
                )
            raise ValueError(
                f"{method_name} for '{context_name}' must return shape "
                f"{expected_shape}, got {output_vector.shape}."
            )

        return output_vector

## 1.4 - Modelling-Stage Examples

These examples focus on how KalmanWorks is meant to be used while defining the physics and geometry of a problem. The goal here is not to run full filter studies yet. The goal is to show how to define a `ProcessModel`, define one or more `MeasurementModel` objects, assemble them into a `StateSpaceModel`, and see what must stay Euclidean by default versus what must be overridden for non-Euclidean state or measurement spaces.


### 1.4.1 - Example 1: 3D Navigation in Euclidean Space

This example is intentionally Euclidean. It shows the baseline modelling workflow:

1. define a process model with Cartesian state and optional control input
2. define measurement models for each sensor channel
3. assemble them into one `StateSpaceModel`
4. construct named state, covariance, control, and measurement vectors
5. inspect the estimator-facing interface

It also shows an important registry concept: two sensors may share the same `MeasurementModel` class while still being registered under different names such as `gps_1` and `gps_2`.


In [ ]:
#EDITED#
class ConstantVelocity3DProcessModel(ProcessModel):
    """
    3D constant-velocity process model in ordinary Cartesian space.

    State
    -----
    x = [pos_x, pos_y, pos_z, vel_x, vel_y, vel_z]

    Optional control
    ----------------
    u = [acc_x, acc_y, acc_z]
    """

    def __init__(self, sigma_a: float) -> None:
        super().__init__(
            state_labels=(
                "pos_x",
                "pos_y",
                "pos_z",
                "vel_x",
                "vel_y",
                "vel_z",
            ),
            control_labels=("acc_x", "acc_y", "acc_z"),
        )
        self.sigma_a = float(sigma_a)

    def F(self, dt: float, x: StateVector | None = None, u: ControlVector | None = None) -> Matrix:
        return np.array(
            [
                [1.0, 0.0, 0.0, dt,  0.0, 0.0],
                [0.0, 1.0, 0.0, 0.0, dt,  0.0],
                [0.0, 0.0, 1.0, 0.0, 0.0, dt ],
                [0.0, 0.0, 0.0, 1.0, 0.0, 0.0],
                [0.0, 0.0, 0.0, 0.0, 1.0, 0.0],
                [0.0, 0.0, 0.0, 0.0, 0.0, 1.0],
            ],
            dtype=np.float64,
        )

    def G(self, dt: float, x: StateVector | None = None, u: ControlVector | None = None) -> Matrix:
        return np.array(
            [
                [0.5 * dt**2, 0.0, 0.0],
                [0.0, 0.5 * dt**2, 0.0],
                [0.0, 0.0, 0.5 * dt**2],
                [dt, 0.0, 0.0],
                [0.0, dt, 0.0],
                [0.0, 0.0, dt],
            ],
            dtype=np.float64,
        )

    def Q(self, dt: float, x: StateVector | None = None, u: ControlVector | None = None) -> Matrix:
        q_1d = (self.sigma_a ** 2) * np.array(
            [
                [dt**4 / 4.0, dt**3 / 2.0],
                [dt**3 / 2.0, dt**2],
            ],
            dtype=np.float64,
        )
        process_noise = np.zeros((6, 6), dtype=np.float64)
        process_noise[np.ix_([0, 3], [0, 3])] = q_1d
        process_noise[np.ix_([1, 4], [1, 4])] = q_1d
        process_noise[np.ix_([2, 5], [2, 5])] = q_1d
        return process_noise


In [ ]:
#EDITED#
class GPSPositionVelocity3DSensorModel(MeasurementModel):
    """
    GPS sensor model.

    Measurement
    -----------
    z = [gps_pos_x, gps_pos_y, gps_pos_z, gps_vel_x, gps_vel_y, gps_vel_z]
    """

    def __init__(self, sigma_pos: float, sigma_vel: float, state_labels: tuple[str, ...]) -> None:
        super().__init__(
            state_labels=state_labels,
            measurement_labels=(
                "gps_pos_x",
                "gps_pos_y",
                "gps_pos_z",
                "gps_vel_x",
                "gps_vel_y",
                "gps_vel_z",
            ),
        )
        self.sigma_pos = float(sigma_pos)
        self.sigma_vel = float(sigma_vel)

    def H(self, x: StateVector | None = None, dt: float | None = None) -> Matrix:
        return np.array(
            [
                [1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
                [0.0, 1.0, 0.0, 0.0, 0.0, 0.0],
                [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
                [0.0, 0.0, 0.0, 1.0, 0.0, 0.0],
                [0.0, 0.0, 0.0, 0.0, 1.0, 0.0],
                [0.0, 0.0, 0.0, 0.0, 0.0, 1.0],
            ],
            dtype=np.float64,
        )

    def R(self, x: StateVector | None = None, z: MeasurementVector | None = None, dt: float | None = None) -> Matrix:
        return np.diag(
            [
                self.sigma_pos ** 2,
                self.sigma_pos ** 2,
                self.sigma_pos ** 2,
                self.sigma_vel ** 2,
                self.sigma_vel ** 2,
                self.sigma_vel ** 2,
            ]
        ).astype(np.float64)


class BarometricAltitudeSensorModel(MeasurementModel):
    """
    Barometric altitude sensor model.

    Measurement
    -----------
    z = [baro_altitude]
    """

    def __init__(self, sigma_altitude: float, state_labels: tuple[str, ...]) -> None:
        super().__init__(
            state_labels=state_labels,
            measurement_labels=("baro_altitude",),
        )
        self.sigma_altitude = float(sigma_altitude)

    def H(self, x: StateVector | None = None, dt: float | None = None) -> Matrix:
        return np.array([[0.0, 0.0, 1.0, 0.0, 0.0, 0.0]], dtype=np.float64)

    def R(self, x: StateVector | None = None, z: MeasurementVector | None = None, dt: float | None = None) -> Matrix:
        return np.array([[self.sigma_altitude ** 2]], dtype=np.float64)


In [ ]:
#EDITED#
# ------------------------------------------------------------
# Step 1: Define the physics components and assemble one system model
# ------------------------------------------------------------
navigation_process_model = ConstantVelocity3DProcessModel(sigma_a=0.20)

gps_sensor_1 = GPSPositionVelocity3DSensorModel(2.0, 0.20, navigation_process_model.state_labels)
gps_sensor_2 = GPSPositionVelocity3DSensorModel(2.5, 0.25, navigation_process_model.state_labels)
barometric_altitude_sensor = BarometricAltitudeSensorModel(3.0, navigation_process_model.state_labels)

navigation_system_model = StateSpaceModel(
    process_model=navigation_process_model,
    measurement_models={
        "gps_1": gps_sensor_1,
        "gps_2": gps_sensor_2,
        "barometric_altitude_sensor": barometric_altitude_sensor,
    },
)

print("=== Euclidean Navigation Model ===")
print("Process model      :", type(navigation_system_model.process_model).__name__)
print("State labels       :", navigation_system_model.state_labels)
print("Measurement names  :", navigation_system_model.measurement_names)
print("gps_1 labels       :", navigation_system_model.get_measurement_labels("gps_1"))
print("gps_2 labels       :", navigation_system_model.get_measurement_labels("gps_2"))
print("Baro labels        :", navigation_system_model.get_measurement_labels("barometric_altitude_sensor"))
print("Control labels     :", navigation_system_model.control_labels)
print("Geometry hooks     : default Euclidean ProcessModel / MeasurementModel behavior")
print("Registry lesson    : gps_1 and gps_2 share one class but use different registration names")


In [ ]:
#EDITED#
# ------------------------------------------------------------
# Step 2: Build named state, covariance, control, and measurements
# ------------------------------------------------------------
x0_nav = navigation_system_model.state(
    pos_x=7000.0,
    pos_y=-1200.0,
    pos_z=1500.0,
    vel_x=120.0,
    vel_y=15.0,
    vel_z=-3.0,
)
P0_nav = navigation_system_model.state_covariance(
    pos_x=50.0**2,
    pos_y=50.0**2,
    pos_z=20.0**2,
    vel_x=5.0**2,
    vel_y=5.0**2,
    vel_z=2.0**2,
)
u0_nav = navigation_system_model.control(acc_x=0.10, acc_y=0.00, acc_z=-0.05)
z_gps_1_nav = navigation_system_model.measurement(
    "gps_1",
    gps_pos_x=7002.3,
    gps_pos_y=-1198.7,
    gps_pos_z=1498.9,
    gps_vel_x=119.6,
    gps_vel_y=15.2,
    gps_vel_z=-2.7,
)
z_gps_2_nav = navigation_system_model.measurement(
    "gps_2",
    gps_pos_x=7001.5,
    gps_pos_y=-1199.4,
    gps_pos_z=1499.8,
    gps_vel_x=120.4,
    gps_vel_y=14.8,
    gps_vel_z=-3.1,
)
z_baro_nav = navigation_system_model.measurement(
    "barometric_altitude_sensor",
    baro_altitude=1496.2,
)

print("=== Named Construction ===")
print("x0_nav =")
print(x0_nav)
print()
print("P0_nav diagonal =")
print(np.diag(P0_nav))
print()
print("u0_nav =", u0_nav)
print("z_gps_1_nav =", z_gps_1_nav)
print("z_gps_2_nav =", z_gps_2_nav)
print("z_baro_nav  =", z_baro_nav)


In [ ]:
#EDITED#
# ------------------------------------------------------------
# Step 3: Inspect the estimator-facing interface
# ------------------------------------------------------------
dt_nav = 0.1
F_nav = navigation_system_model.F(dt=dt_nav, x=x0_nav, u=u0_nav)
Q_nav = navigation_system_model.Q(dt=dt_nav, x=x0_nav, u=u0_nav)
G_nav = navigation_system_model.G(dt=dt_nav, x=x0_nav, u=u0_nav)
H_gps_1_nav = navigation_system_model.H(x=x0_nav, measurement_name="gps_1", dt=dt_nav)
R_gps_1_nav = navigation_system_model.R(x=x0_nav, z=z_gps_1_nav, measurement_name="gps_1", dt=dt_nav)
H_gps_2_nav = navigation_system_model.H(x=x0_nav, measurement_name="gps_2", dt=dt_nav)
R_gps_2_nav = navigation_system_model.R(x=x0_nav, z=z_gps_2_nav, measurement_name="gps_2", dt=dt_nav)
H_baro_nav = navigation_system_model.H(x=x0_nav, measurement_name="barometric_altitude_sensor", dt=dt_nav)
R_baro_nav = navigation_system_model.R(x=x0_nav, z=z_baro_nav, measurement_name="barometric_altitude_sensor", dt=dt_nav)

print("=== Named Lookups and Estimator Access ===")
print("Position indices   ->", navigation_system_model.state_indices("pos_x", "pos_y", "pos_z"))
print("Velocity indices   ->", navigation_system_model.state_indices("vel_x", "vel_y", "vel_z"))
print("Control indices    ->", navigation_system_model.control_indices("acc_x", "acc_y", "acc_z"))
print("gps_1 meas idx     ->", navigation_system_model.measurement_indices("gps_1", "gps_pos_x", "gps_vel_z"))
print("gps_2 meas idx     ->", navigation_system_model.measurement_indices("gps_2", "gps_pos_y", "gps_vel_x"))
print("F/Q/G shapes       ->", F_nav.shape, Q_nav.shape, G_nav.shape)
print("H(gps_1), R(gps_1) ->", H_gps_1_nav.shape, R_gps_1_nav.shape)
print("H(gps_2), R(gps_2) ->", H_gps_2_nav.shape, R_gps_2_nav.shape)
print("H(baro),  R(baro)  ->", H_baro_nav.shape, R_baro_nav.shape)


In [ ]:
#EDITED#
class BadNavigationSensorModel(MeasurementModel):
    """Deliberately incompatible sensor model for fail-fast demonstration."""

    def __init__(self) -> None:
        super().__init__(
            state_labels=("pos_x", "pos_y", "vel_x", "vel_y"),
            measurement_labels=("bad_altitude",),
        )

    def R(self, x: StateVector | None = None, z: MeasurementVector | None = None, dt: float | None = None) -> Matrix:
        return np.array([[1.0]], dtype=np.float64)


print("=== Fail-Fast Checks ===")
try:
    navigation_system_model.measurement(
        gps_pos_x=7002.3,
        gps_pos_y=-1198.7,
        gps_pos_z=1498.9,
        gps_vel_x=119.6,
        gps_vel_y=15.2,
        gps_vel_z=-2.7,
    )
except ValueError as error:
    print("Missing measurement_name in multi-sensor model ->", error)

try:
    navigation_system_model.add_measurement_model("gps_1", gps_sensor_1)
except ValueError as error:
    print("Duplicate registration name ->", error)

try:
    StateSpaceModel(
        process_model=navigation_process_model,
        measurement_models={"bad_navigation_sensor": BadNavigationSensorModel()},
    )
except ValueError as error:
    print("State-layout mismatch at assembly ->", error)


### 1.4.2 - Example 2: Attitude State with Quaternion Geometry Hooks

This example is still about model construction, but now the state is not purely Euclidean. The important modelling lesson is that the process model must take responsibility for geometry-sensitive state operations such as:

- `apply_state_delta(...)`
- `state_residual(...)`
- `state_mean(...)`

The quaternion helpers below are intentionally placeholders. They are here to mark the modelling boundary clearly, not to pretend quaternion algebra is solved in Section 1.


In [ ]:
#EDITED#
def quat_normalize(q: ArrayLike) -> NDArray[np.float64]:
    """Return q normalized to unit length."""
    quat = np.asarray(q, dtype=np.float64)
    norm = np.linalg.norm(quat)
    if norm <= 0.0:
        raise ValueError("Quaternion norm must be positive.")
    return quat / norm


def quat_with_positive_scalar(q: ArrayLike) -> NDArray[np.float64]:
    """Flip sign so the scalar component stays non-negative."""
    quat = quat_normalize(q)
    return -quat if quat[0] < 0.0 else quat


def quat_addition(q: ArrayLike, dq: ArrayLike) -> NDArray[np.float64]:
    """Placeholder quaternion update; replace with your own library helper."""
    raise NotImplementedError(
        "Quaternion state updates are convention-specific. Replace quat_addition(...) "
        "with your own trusted quaternion helper."
    )


def quat_subtraction(q2: ArrayLike, q1: ArrayLike) -> NDArray[np.float64]:
    """Placeholder quaternion residual; replace with your own library helper."""
    raise NotImplementedError(
        "Quaternion residuals are convention-specific. Replace quat_subtraction(...) "
        "with your own trusted quaternion helper."
    )


def quat_mean(samples: ArrayLike, weights: ArrayLike) -> NDArray[np.float64]:
    """Placeholder quaternion mean; replace with your own library helper."""
    raise NotImplementedError(
        "Quaternion averaging is convention-specific. Replace quat_mean(...) "
        "with your own trusted quaternion helper."
    )


In [ ]:
#EDITED#
class AttitudeProcessModel(ProcessModel):
    """
    Attitude process model with quaternion attitude and Euclidean body-rate states.

    State
    -----
    x = [q1, q2, q3, q4, omega_x, omega_y, omega_z]
    """

    def __init__(self, sigma_quat: float, sigma_omega: float) -> None:
        super().__init__(
            state_labels=("q1", "q2", "q3", "q4", "omega_x", "omega_y", "omega_z")
        )
        self.sigma_quat = float(sigma_quat)
        self.sigma_omega = float(sigma_omega)

    def Q(self, dt: float, x: StateVector | None = None, u: ControlVector | None = None) -> Matrix:
        return np.diag(
            [
                self.sigma_quat ** 2,
                self.sigma_quat ** 2,
                self.sigma_quat ** 2,
                self.sigma_quat ** 2,
                self.sigma_omega ** 2,
                self.sigma_omega ** 2,
                self.sigma_omega ** 2,
            ]
        ).astype(np.float64)

    def apply_state_delta(self, x: StateVector, dx: StateVector) -> StateVector:
        state = np.asarray(x, dtype=np.float64)
        delta = np.asarray(dx, dtype=np.float64)
        updated_state = np.empty_like(state)
        updated_state[:4] = quat_with_positive_scalar(quat_addition(state[:4], delta[:4]))
        updated_state[4:] = state[4:] + delta[4:]
        return updated_state

    def state_residual(self, a: StateVector, b: StateVector) -> StateVector:
        state_a = np.asarray(a, dtype=np.float64)
        state_b = np.asarray(b, dtype=np.float64)
        residual = np.empty_like(state_a)
        residual[:4] = quat_subtraction(state_a[:4], state_b[:4])
        residual[4:] = state_a[4:] - state_b[4:]
        return residual

    def state_mean(self, samples: SampleMatrix, weights: WeightVector) -> StateVector:
        state_samples = np.asarray(samples, dtype=np.float64)
        sample_weights = np.asarray(weights, dtype=np.float64)
        mean_state = np.empty(state_samples.shape[1], dtype=np.float64)
        mean_state[:4] = quat_with_positive_scalar(quat_mean(state_samples[:, :4], sample_weights))
        mean_state[4:] = np.average(state_samples[:, 4:], axis=0, weights=sample_weights)
        return mean_state


class GyroSensorModel(MeasurementModel):
    """
    Gyroscope measurement model for Euclidean body-rate states.

    Measurement
    -----------
    z = [gyro_omega_x, gyro_omega_y, gyro_omega_z]
    """

    def __init__(self, sigma_gyro: float, state_labels: tuple[str, ...]) -> None:
        super().__init__(
            state_labels=state_labels,
            measurement_labels=("gyro_omega_x", "gyro_omega_y", "gyro_omega_z"),
        )
        self.sigma_gyro = float(sigma_gyro)

    def H(self, x: StateVector | None = None, dt: float | None = None) -> Matrix:
        return np.array(
            [
                [0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0],
                [0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0],
                [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0],
            ],
            dtype=np.float64,
        )

    def R(self, x: StateVector | None = None, z: MeasurementVector | None = None, dt: float | None = None) -> Matrix:
        return np.diag([self.sigma_gyro ** 2] * 3).astype(np.float64)


class StarTrackerSensorModel(MeasurementModel):
    """
    Star tracker measurement model for quaternion attitude.

    Measurement
    -----------
    z = [star_tracker_q1, star_tracker_q2, star_tracker_q3, star_tracker_q4]
    """

    def __init__(self, sigma_star_tracker: float, state_labels: tuple[str, ...]) -> None:
        super().__init__(
            state_labels=state_labels,
            measurement_labels=(
                "star_tracker_q1",
                "star_tracker_q2",
                "star_tracker_q3",
                "star_tracker_q4",
            ),
        )
        self.sigma_star_tracker = float(sigma_star_tracker)

    def h(self, x: StateVector, dt: float | None = None) -> MeasurementVector:
        state = np.asarray(x, dtype=np.float64)
        return quat_with_positive_scalar(state[:4])

    def R(self, x: StateVector | None = None, z: MeasurementVector | None = None, dt: float | None = None) -> Matrix:
        return np.diag([self.sigma_star_tracker ** 2] * 4).astype(np.float64)

    def innovation(self, z: MeasurementVector, z_pred: MeasurementVector) -> MeasurementVector:
        measured_quaternion = np.asarray(z, dtype=np.float64)
        predicted_quaternion = np.asarray(z_pred, dtype=np.float64)
        return quat_subtraction(measured_quaternion, predicted_quaternion)


In [ ]:
#EDITED#
# ------------------------------------------------------------
# Step 1: Assemble the attitude model
# ------------------------------------------------------------
attitude_process_model = AttitudeProcessModel(sigma_quat=1.0e-4, sigma_omega=1.0e-2)
gyro_sensor = GyroSensorModel(0.005, attitude_process_model.state_labels)
star_tracker_sensor = StarTrackerSensorModel(0.001, attitude_process_model.state_labels)

attitude_system_model = StateSpaceModel(
    process_model=attitude_process_model,
    measurement_models={
        "gyro_sensor": gyro_sensor,
        "star_tracker_sensor": star_tracker_sensor,
    },
)

x0_attitude = attitude_system_model.state(
    q1=1.0,
    q2=0.0,
    q3=0.0,
    q4=0.0,
    omega_x=0.01,
    omega_y=-0.02,
    omega_z=0.015,
)
P0_attitude = attitude_system_model.state_covariance(
    q1=1.0e-6,
    q2=1.0e-6,
    q3=1.0e-6,
    q4=1.0e-6,
    omega_x=1.0e-4,
    omega_y=1.0e-4,
    omega_z=1.0e-4,
)
z_gyro_attitude = attitude_system_model.measurement(
    "gyro_sensor",
    gyro_omega_x=0.0105,
    gyro_omega_y=-0.0197,
    gyro_omega_z=0.0148,
)
z_star_tracker_attitude = attitude_system_model.measurement(
    "star_tracker_sensor",
    star_tracker_q1=0.9999,
    star_tracker_q2=0.0010,
    star_tracker_q3=-0.0020,
    star_tracker_q4=0.0040,
)

dt_attitude = 0.1
Q_attitude = attitude_system_model.Q(dt=dt_attitude, x=x0_attitude)
H_gyro_attitude = attitude_system_model.H(x=x0_attitude, measurement_name="gyro_sensor", dt=dt_attitude)
R_gyro_attitude = attitude_system_model.R(
    x=x0_attitude,
    z=z_gyro_attitude,
    measurement_name="gyro_sensor",
    dt=dt_attitude,
)
z_pred_star_tracker_attitude = attitude_system_model.h(
    x=x0_attitude,
    measurement_name="star_tracker_sensor",
    dt=dt_attitude,
)
R_star_tracker_attitude = attitude_system_model.R(
    x=x0_attitude,
    z=z_star_tracker_attitude,
    measurement_name="star_tracker_sensor",
    dt=dt_attitude,
)

print("=== Attitude Model with Geometry Hooks ===")
print("State labels          :", attitude_system_model.state_labels)
print("Quaternion indices    :", attitude_system_model.state_indices("q1", "q2", "q3", "q4"))
print("Angular-rate indices  :", attitude_system_model.state_indices("omega_x", "omega_y", "omega_z"))
print("Measurement names     :", attitude_system_model.measurement_names)
print("Q shape               :", Q_attitude.shape)
print("H(gyro), R(gyro)      :", H_gyro_attitude.shape, R_gyro_attitude.shape)
print("h(star_tracker) shape :", z_pred_star_tracker_attitude.shape)
print("R(star_tracker) shape :", R_star_tracker_attitude.shape)
print("Geometry lesson       : quaternion state hooks are overridden by the process model")
print("Measurement lesson    : star_tracker_sensor also owns non-Euclidean innovation logic")
print("P0_attitude diagonal  :", np.diag(P0_attitude))


In [ ]:
#EDITED#
# ------------------------------------------------------------
# Step 2: Show the geometry-aware hooks without faking quaternion math
# ------------------------------------------------------------
x_trial_attitude = x0_attitude.copy()
dx_trial_attitude = np.array([0.0, 0.001, -0.002, 0.003, 0.0005, -0.0005, 0.0002], dtype=np.float64)
sample_states_attitude = np.vstack(
    [
        x0_attitude,
        attitude_system_model.state(
            q1=0.9998,
            q2=0.0010,
            q3=0.0000,
            q4=-0.0200,
            omega_x=0.0110,
            omega_y=-0.0210,
            omega_z=0.0155,
        ),
    ]
)
sample_weights_attitude = np.array([0.6, 0.4], dtype=np.float64)

print("=== Geometry-Hook Demonstration ===")
for label, operation in [
    (
        "apply_state_delta(...)",
        lambda: attitude_system_model.apply_state_delta(x_trial_attitude, dx_trial_attitude),
    ),
    (
        "state_residual(...)",
        lambda: attitude_system_model.state_residual(sample_states_attitude[1], sample_states_attitude[0]),
    ),
    (
        "state_mean(...)",
        lambda: attitude_system_model.state_mean(sample_states_attitude, sample_weights_attitude),
    ),
    (
        "innovation(...) for star_tracker_sensor",
        lambda: attitude_system_model.innovation(
            z=z_star_tracker_attitude,
            z_pred=z_pred_star_tracker_attitude,
            measurement_name="star_tracker_sensor",
        ),
    ),
]:
    try:
        result = operation()
        print(label, "->", result)
    except NotImplementedError as error:
        print(label, "-> placeholder reached:", error)


In [ ]:
#EDITED#
class BrokenAttitudeProcessModel(ProcessModel):
    """Deliberately invalid state labels for fail-fast demonstration."""

    def __init__(self) -> None:
        super().__init__(
            state_labels=("q1", "q2", "q3", "q4", "omega_x", "omega y", "omega_z")
        )

    def Q(self, dt: float, x: StateVector | None = None, u: ControlVector | None = None) -> Matrix:
        return np.eye(7, dtype=np.float64)


class BrokenStarTrackerLabels(MeasurementModel):
    """Deliberately invalid measurement labels for fail-fast demonstration."""

    def __init__(self, state_labels: tuple[str, ...]) -> None:
        super().__init__(
            state_labels=state_labels,
            measurement_labels=("star_tracker_q1", "star tracker_q2", "star_tracker_q3", "star_tracker_q4"),
        )

    def R(self, x: StateVector | None = None, z: MeasurementVector | None = None, dt: float | None = None) -> Matrix:
        return np.eye(4, dtype=np.float64)


class BadAttitudeSensorModel(MeasurementModel):
    """Deliberately incompatible measurement model for fail-fast demonstration."""

    def __init__(self) -> None:
        super().__init__(
            state_labels=("q1", "q2", "q3", "omega_x", "omega_y", "omega_z"),
            measurement_labels=("bad_star_tracker_q1",),
        )

    def R(self, x: StateVector | None = None, z: MeasurementVector | None = None, dt: float | None = None) -> Matrix:
        return np.array([[1.0]], dtype=np.float64)


print("=== Fail-Fast Checks ===")
try:
    BrokenAttitudeProcessModel()
except ValueError as error:
    print("Bad state label definition ->", error)

try:
    BrokenStarTrackerLabels(attitude_process_model.state_labels)
except ValueError as error:
    print("Bad measurement label definition ->", error)

try:
    StateSpaceModel(
        process_model=attitude_process_model,
        measurement_models={"bad_attitude_sensor": BadAttitudeSensorModel()},
    )
except ValueError as error:
    print("State-layout mismatch at assembly ->", error)


### 1.4.3 - Example 3: Mixed Euclidean + Non-Euclidean State

Many practical systems mix ordinary Euclidean states with non-Euclidean states. This example uses Cartesian position together with quaternion attitude. The modelling lesson is simple:

- Euclidean state blocks can keep ordinary addition and subtraction
- non-Euclidean state blocks must override the geometry hooks
- measurement models may also need custom `innovation(...)` logic when the measurement space is not Euclidean, such as wrapped angles or bearings


In [ ]:
#EDITED#
def wrap_angle(angle_rad: float) -> float:
    """Wrap one angle to the interval [-pi, pi)."""
    return float((angle_rad + np.pi) % (2.0 * np.pi) - np.pi)


class MixedPoseProcessModel(ProcessModel):
    """
    Mixed process model with Euclidean position and quaternion attitude.

    State
    -----
    x = [pos_x, pos_y, pos_z, q1, q2, q3, q4]
    """

    def __init__(self, sigma_pos: float, sigma_quat: float) -> None:
        super().__init__(
            state_labels=("pos_x", "pos_y", "pos_z", "q1", "q2", "q3", "q4")
        )
        self.sigma_pos = float(sigma_pos)
        self.sigma_quat = float(sigma_quat)

    def F(self, dt: float, x: StateVector | None = None, u: ControlVector | None = None) -> Matrix:
        return np.eye(self.state_dim, dtype=np.float64)

    def Q(self, dt: float, x: StateVector | None = None, u: ControlVector | None = None) -> Matrix:
        return np.diag(
            [
                self.sigma_pos ** 2,
                self.sigma_pos ** 2,
                self.sigma_pos ** 2,
                self.sigma_quat ** 2,
                self.sigma_quat ** 2,
                self.sigma_quat ** 2,
                self.sigma_quat ** 2,
            ]
        ).astype(np.float64)

    def apply_state_delta(self, x: StateVector, dx: StateVector) -> StateVector:
        state = np.asarray(x, dtype=np.float64)
        delta = np.asarray(dx, dtype=np.float64)
        updated_state = np.empty_like(state)
        updated_state[:3] = state[:3] + delta[:3]
        updated_state[3:] = quat_with_positive_scalar(quat_addition(state[3:], delta[3:]))
        return updated_state

    def state_residual(self, a: StateVector, b: StateVector) -> StateVector:
        state_a = np.asarray(a, dtype=np.float64)
        state_b = np.asarray(b, dtype=np.float64)
        residual = np.empty_like(state_a)
        residual[:3] = state_a[:3] - state_b[:3]
        residual[3:] = quat_subtraction(state_a[3:], state_b[3:])
        return residual

    def state_mean(self, samples: SampleMatrix, weights: WeightVector) -> StateVector:
        state_samples = np.asarray(samples, dtype=np.float64)
        sample_weights = np.asarray(weights, dtype=np.float64)
        mean_state = np.empty(state_samples.shape[1], dtype=np.float64)
        mean_state[:3] = np.average(state_samples[:, :3], axis=0, weights=sample_weights)
        mean_state[3:] = quat_with_positive_scalar(quat_mean(state_samples[:, 3:], sample_weights))
        return mean_state


In [ ]:
#EDITED#
class MixedPositionSensorModel(MeasurementModel):
    """
    Cartesian position sensor model using default Euclidean innovation.

    Measurement
    -----------
    z = [meas_pos_x, meas_pos_y, meas_pos_z]
    """

    def __init__(self, sigma_position: float, state_labels: tuple[str, ...]) -> None:
        super().__init__(
            state_labels=state_labels,
            measurement_labels=("meas_pos_x", "meas_pos_y", "meas_pos_z"),
        )
        self.sigma_position = float(sigma_position)

    def H(self, x: StateVector | None = None, dt: float | None = None) -> Matrix:
        return np.array(
            [
                [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
                [0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
                [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0],
            ],
            dtype=np.float64,
        )

    def R(self, x: StateVector | None = None, z: MeasurementVector | None = None, dt: float | None = None) -> Matrix:
        return np.diag([self.sigma_position ** 2] * 3).astype(np.float64)


class AzimuthSensorModel(MeasurementModel):
    """
    Wrapped-angle azimuth sensor model.

    Measurement
    -----------
    z = [azimuth]
    """

    def __init__(self, sigma_azimuth: float, state_labels: tuple[str, ...]) -> None:
        super().__init__(
            state_labels=state_labels,
            measurement_labels=("azimuth",),
        )
        self.sigma_azimuth = float(sigma_azimuth)

    def h(self, x: StateVector, dt: float | None = None) -> MeasurementVector:
        state = np.asarray(x, dtype=np.float64)
        return np.array([np.arctan2(state[1], state[0])], dtype=np.float64)

    def R(self, x: StateVector | None = None, z: MeasurementVector | None = None, dt: float | None = None) -> Matrix:
        return np.array([[self.sigma_azimuth ** 2]], dtype=np.float64)

    def innovation(self, z: MeasurementVector, z_pred: MeasurementVector) -> MeasurementVector:
        measured = np.asarray(z, dtype=np.float64)
        predicted = np.asarray(z_pred, dtype=np.float64)
        return np.array([wrap_angle(measured[0] - predicted[0])], dtype=np.float64)


In [ ]:
#EDITED#
# ------------------------------------------------------------
# Step 1: Assemble a mixed Euclidean + non-Euclidean system model
# ------------------------------------------------------------
mixed_pose_process_model = MixedPoseProcessModel(sigma_pos=0.10, sigma_quat=1.0e-4)
mixed_position_sensor = MixedPositionSensorModel(0.50, mixed_pose_process_model.state_labels)
azimuth_sensor = AzimuthSensorModel(np.deg2rad(1.0), mixed_pose_process_model.state_labels)

mixed_pose_system_model = StateSpaceModel(
    process_model=mixed_pose_process_model,
    measurement_models={
        "position_sensor": mixed_position_sensor,
        "azimuth_sensor": azimuth_sensor,
    },
)

x0_mixed = mixed_pose_system_model.state(
    pos_x=-10.0,
    pos_y=0.2,
    pos_z=2.0,
    q1=1.0,
    q2=0.0,
    q3=0.0,
    q4=0.0,
)
P0_mixed = mixed_pose_system_model.state_covariance(
    pos_x=0.5**2,
    pos_y=0.5**2,
    pos_z=0.2**2,
    q1=1.0e-6,
    q2=1.0e-6,
    q3=1.0e-6,
    q4=1.0e-6,
)
z_position_mixed = mixed_pose_system_model.measurement(
    "position_sensor",
    meas_pos_x=-9.7,
    meas_pos_y=0.4,
    meas_pos_z=2.1,
)
z_azimuth_mixed = mixed_pose_system_model.measurement(
    "azimuth_sensor",
    azimuth=-3.13,
)

dt_mixed = 0.1
F_mixed = mixed_pose_system_model.F(dt=dt_mixed, x=x0_mixed)
Q_mixed = mixed_pose_system_model.Q(dt=dt_mixed, x=x0_mixed)
H_position_mixed = mixed_pose_system_model.H(
    x=x0_mixed,
    measurement_name="position_sensor",
    dt=dt_mixed,
)
R_position_mixed = mixed_pose_system_model.R(
    x=x0_mixed,
    z=z_position_mixed,
    measurement_name="position_sensor",
    dt=dt_mixed,
)
z_pred_position_mixed = H_position_mixed @ x0_mixed
innovation_position_mixed = mixed_pose_system_model.innovation(
    z=z_position_mixed,
    z_pred=z_pred_position_mixed,
    measurement_name="position_sensor",
)
z_pred_azimuth_mixed = mixed_pose_system_model.h(
    x=x0_mixed,
    measurement_name="azimuth_sensor",
    dt=dt_mixed,
)
innovation_azimuth_mixed = mixed_pose_system_model.innovation(
    z=z_azimuth_mixed,
    z_pred=z_pred_azimuth_mixed,
    measurement_name="azimuth_sensor",
)

print("=== Mixed-State Modelling Choices ===")
print("State labels               :", mixed_pose_system_model.state_labels)
print("Position block indices     :", mixed_pose_system_model.state_indices("pos_x", "pos_y", "pos_z"))
print("Quaternion block indices   :", mixed_pose_system_model.state_indices("q1", "q2", "q3", "q4"))
print("Measurement names          :", mixed_pose_system_model.measurement_names)
print("Process geometry split     : Euclidean position block + quaternion geometry block")
print("Measurement geometry split : position_sensor uses default innovation; azimuth_sensor wraps the residual")
print("F/Q shapes                 :", F_mixed.shape, Q_mixed.shape)
print("H(position), R(position)   :", H_position_mixed.shape, R_position_mixed.shape)
print("Position innovation        :", innovation_position_mixed)
print("Predicted azimuth          :", z_pred_azimuth_mixed)
print("Wrapped azimuth innovation :", innovation_azimuth_mixed)
print("P0_mixed diagonal          :", np.diag(P0_mixed))


In [ ]:
#EDITED#
class BrokenAzimuthLabels(MeasurementModel):
    """Deliberately invalid measurement labels for fail-fast demonstration."""

    def __init__(self, state_labels: tuple[str, ...]) -> None:
        super().__init__(
            state_labels=state_labels,
            measurement_labels=("azimuth", "bearing-angle"),
        )

    def R(self, x: StateVector | None = None, z: MeasurementVector | None = None, dt: float | None = None) -> Matrix:
        return np.eye(2, dtype=np.float64)


class BadMixedSensorModel(MeasurementModel):
    """Deliberately incompatible measurement model for fail-fast demonstration."""

    def __init__(self) -> None:
        super().__init__(
            state_labels=("pos_x", "pos_y", "pos_z", "q1", "q2", "q3"),
            measurement_labels=("bad_meas_pos_x",),
        )

    def R(self, x: StateVector | None = None, z: MeasurementVector | None = None, dt: float | None = None) -> Matrix:
        return np.array([[1.0]], dtype=np.float64)


print("=== Fail-Fast Checks ===")
try:
    BrokenAzimuthLabels(mixed_pose_process_model.state_labels)
except ValueError as error:
    print("Bad measurement label definition ->", error)

try:
    StateSpaceModel(
        process_model=mixed_pose_process_model,
        measurement_models={"bad_mixed_sensor": BadMixedSensorModel()},
    )
except ValueError as error:
    print("State-layout mismatch at assembly ->", error)
